# Custom Food Detection Model Training

**Training YOLOv8 for Food Recognition**

This notebook will guide you through training a custom YOLOv8 model to detect food items. This trained model can then be integrated with the Food Vision System to recognize many more foods than the default COCO model.

## What You'll Learn

1. How to prepare a food detection dataset in YOLO format
2. How to train a YOLOv8 model on your custom dataset
3. How to evaluate and test your trained model
4. How to integrate the trained model into the Food Vision System

## Prerequisites

- Food images with bounding box annotations
- Dataset in YOLO format (see dataset preparation guide)
- Google Colab Pro recommended (free tier works but slower)


## Step 1: Install Required Packages


In [ ]:
# Install Ultralytics YOLOv8 and other required packages
!pip install ultralytics roboflow opencv-python-headless matplotlib -q

print("✅ Installation complete!")


## Step 2: Dataset Options

You have several options for getting a food detection dataset:

### Option A: Use a Public Dataset (Recommended for Beginners)

- **Roboflow Universe**: Pre-annotated food datasets
- **Food-101**: 101 food categories (classification, needs conversion)
- **UECFood100**: 100 Japanese food items with bounding boxes

### Option B: Create Your Own Dataset

- Collect food images
- Annotate with bounding boxes using tools like:
  - **Roboflow** (online, free tier available)
  - **LabelImg** (desktop application)
  - **CVAT** (web-based)

### Option C: Use Synthetic Data

- Generate synthetic food images
- Use augmentation techniques


## Step 3A: Load Dataset from Roboflow (Easy Method)

If you have a dataset on Roboflow, use this cell to download it.


In [ ]:
# ============================================================================
# ROBOFLOW DATASET LOADING
# ============================================================================
# Fill in your Roboflow API details below:
#
# IMPORTANT: To find your correct workspace name:
# 1. Go to your Roboflow project: https://app.roboflow.com/
# 2. Click on your project "FoodVision-Detection"
# 3. Look at the URL in your browser
# 4. The URL format is: https://app.roboflow.com/[workspace-name]/[project-name]/[version]
# 5. Copy the workspace name EXACTLY as it appears (case-sensitive!)

from roboflow import Roboflow

# ⬇️ ENTER YOUR ROBOFLOW DETAILS HERE ⬇️
ROBOFLOW_API_KEY = "rf_e6bBzSGyJBUXPkgFtk9ba1Zabdp2"        # ✅ API Key configured
ROBOFLOW_WORKSPACE = "Meghashyam"                # ⚠️ UPDATE THIS - Check your Roboflow URL for exact name!
ROBOFLOW_PROJECT = "foodvision-detection-acvpj"  # ✅ Project slug/ID
ROBOFLOW_VERSION = 1                             # ✅ Version number (check in Roboflow dashboard)

# Load dataset from Roboflow
try:
    print(f"🔄 Connecting to Roboflow...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    print(f"📁 Accessing workspace: {ROBOFLOW_WORKSPACE}")
    print(f"📦 Accessing project: {ROBOFLOW_PROJECT}")
    print(f"🔢 Version: {ROBOFLOW_VERSION}")
    
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(ROBOFLOW_VERSION).download("yolov8")
    
    DATASET_PATH = dataset.location
    print(f"\n✅ Dataset downloaded successfully!")
    print(f"📁 Dataset location: {DATASET_PATH}")
    
    # Verify dataset structure
    from pathlib import Path
    train_path = Path(DATASET_PATH) / "train" / "images"
    val_path = Path(DATASET_PATH) / "valid" / "images"
    
    if train_path.exists():
        train_images = len(list(train_path.glob("*.jpg")) + list(train_path.glob("*.png")))
        print(f"📊 Training images: {train_images}")
    
    if val_path.exists():
        val_images = len(list(val_path.glob("*.jpg")) + list(val_path.glob("*.png")))
        print(f"📊 Validation images: {val_images}")
        
except Exception as e:
    print(f"\n❌ Error loading dataset from Roboflow: {e}")
    print("\n" + "="*70)
    print("💡 HOW TO FIX THIS:")
    print("="*70)
    print("\n1. Go to your Roboflow project in your browser:")
    print("   https://app.roboflow.com/")
    print("\n2. Click on your project 'FoodVision-Detection'")
    print("\n3. Look at the URL - it will look like this:")
    print("   https://app.roboflow.com/YOUR-WORKSPACE-NAME/FoodVision-Detection/1")
    print("\n4. Copy the workspace name from the URL (the part after /com/ and before /FoodVision-Detection)")
    print("\n5. Update ROBOFLOW_WORKSPACE in this cell with the EXACT name from the URL")
    print("   (Workspace names are case-sensitive!)")
    print("\n6. Also verify:")
    print("   - Project name matches exactly (case-sensitive)")
    print("   - Version number exists in your project")
    print("   - You have access permissions to the dataset")
    print("\n" + "="*70)
    print("\n⚠️ Falling back to manual dataset path...")
    DATASET_PATH = "/content/datasets"  # Fallback path


## Step 3B: Upload Your Own Dataset

If you have your own annotated dataset, upload it to Colab.


In [ ]:
# Dataset Structure Required:
# 
# dataset/
# ├── train/
# │   ├── images/
# │   │   ├── image1.jpg
# │   │   └── image2.jpg
# │   └── labels/
# │       ├── image1.txt
# │       └── image2.txt
# ├── val/
# │   ├── images/
# │   └── labels/
# └── test/ (optional)
#     ├── images/
#     └── labels/
#
# Label format (YOLO): class_id center_x center_y width height (normalized 0-1)
# Example: 0 0.5 0.5 0.3 0.4

import os
from pathlib import Path

# Set your dataset path here
DATASET_PATH = "/content/datasets/food_dataset"  # Update this

# Check if dataset exists
if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found at: {DATASET_PATH}")
    
    # Count images
    train_images = len(list(Path(DATASET_PATH, "train/images").glob("*.jpg"))) + \
                   len(list(Path(DATASET_PATH, "train/images").glob("*.png")))
    val_images = len(list(Path(DATASET_PATH, "val/images").glob("*.jpg"))) + \
                 len(list(Path(DATASET_PATH, "val/images").glob("*.png")))
    
    print(f"Training images: {train_images}")
    print(f"Validation images: {val_images}")
else:
    print(f"⚠️ Dataset not found at: {DATASET_PATH}")
    print("Please upload your dataset or update DATASET_PATH")
    print("\nTo upload:")
    print("1. Zip your dataset folder")
    print("2. Click the folder icon in the left sidebar")
    print("3. Upload the zip file")
    print("4. Unzip it in a code cell: !unzip dataset.zip -d /content/datasets/")


## Step 4: Create Dataset Configuration File

Create a `data.yaml` file that tells YOLOv8 about your dataset structure and classes.


In [ ]:
# Define your food classes
# Update this list with the foods you want to detect

FOOD_CLASSES = [
    'apple',
    'banana',
    'pizza',
    'hamburger',
    'sandwich',
    'rice',
    'pasta',
    'chicken',
    'fish',
    'salad',
    'soup',
    'bread',
    'cake',
    'cookie',
    'donut',
    # Add more food classes as needed
]

# Verify dataset exists before creating config
from pathlib import Path
import os

# Check if dataset path exists
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"❌ Dataset path not found: {DATASET_PATH}\n"
        f"Please make sure the Roboflow download completed successfully, "
        f"or upload your dataset manually."
    )

# Check dataset structure - Roboflow uses 'valid' folder, not 'val'
train_path = Path(DATASET_PATH) / "train" / "images"
valid_path = Path(DATASET_PATH) / "valid" / "images"
val_path = Path(DATASET_PATH) / "val" / "images"

# Determine which validation folder exists
if valid_path.exists():
    val_folder = "valid"
elif val_path.exists():
    val_folder = "val"
else:
    raise FileNotFoundError(
        f"❌ Validation folder not found in {DATASET_PATH}\n"
        f"Expected either 'valid/images' or 'val/images' folder."
    )

if not train_path.exists():
    raise FileNotFoundError(
        f"❌ Training folder not found: {train_path}\n"
        f"Please check your dataset structure."
    )

print(f"✅ Dataset structure verified!")
print(f"📁 Dataset path: {DATASET_PATH}")
print(f"📊 Training images: {len(list(train_path.glob('*.jpg')) + list(train_path.glob('*.png')))}")
val_img_path = Path(DATASET_PATH) / val_folder / "images"
print(f"📊 Validation images: {len(list(val_img_path.glob('*.jpg')) + list(val_img_path.glob('*.png')))}")

# Create data.yaml configuration file
yaml_content = f"""# Food Detection Dataset Configuration
path: {DATASET_PATH}  # dataset root dir
train: train/images  # train images (relative to 'path')
val: {val_folder}/images  # validation images (relative to 'path')
test: test/images    # test images (optional, relative to 'path')

# Classes
nc: {len(FOOD_CLASSES)}  # number of classes
names: {FOOD_CLASSES}  # class names

# Additional info
description: Custom Food Detection Dataset for YOLOv8
"""

# Save the configuration file
config_path = "/content/data.yaml"
with open(config_path, 'w') as f:
    f.write(yaml_content)

print("✅ Dataset configuration file created!")
print(f"📄 Config saved to: {config_path}")
print(f"\nClasses ({len(FOOD_CLASSES)}):")
for i, cls in enumerate(FOOD_CLASSES):
    print(f"  {i}: {cls}")

# Display the config file
print("\n" + "="*50)
print("Configuration file contents:")
print("="*50)
print(yaml_content)


In [ ]:
from ultralytics import YOLO
import torch

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Initialize model
# Start from pre-trained weights for better results (transfer learning)
MODEL_SIZE = "yolov8n"  # Options: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x

model = YOLO(f'{MODEL_SIZE}.pt')  # Load pre-trained weights

print(f"✅ Model initialized: {MODEL_SIZE}.pt")
print(f"Number of classes in model: {len(FOOD_CLASSES)}")


## Step 6: Configure Training Parameters

Customize these parameters based on your dataset size and requirements.


In [ ]:
# Training parameters
TRAINING_CONFIG = {
    # Basic settings
    'epochs': 100,              # Number of training epochs
    'imgsz': 640,               # Image size for training
    'batch': 16,                # Batch size (reduce if out of memory)
    'workers': 4,               # Number of data loading workers
    
    # Learning rate
    'lr0': 0.01,                # Initial learning rate
    'lrf': 0.01,                # Final learning rate (lr0 * lrf)
    
    # Optimization
    'momentum': 0.937,          # SGD momentum
    'weight_decay': 0.0005,     # Weight decay
    
    # Augmentation
    'hsv_h': 0.015,             # HSV-Hue augmentation
    'hsv_s': 0.7,               # HSV-Saturation augmentation
    'hsv_v': 0.4,               # HSV-Value augmentation
    'degrees': 0.0,             # Rotation augmentation
    'translate': 0.1,           # Translation augmentation
    'scale': 0.5,               # Scale augmentation
    'flipud': 0.0,              # Vertical flip probability
    'fliplr': 0.5,              # Horizontal flip probability
    'mosaic': 1.0,              # Mosaic augmentation probability
    'mixup': 0.0,               # Mixup augmentation probability
    
    # Other settings
    'patience': 50,             # Early stopping patience
    'save': True,               # Save checkpoints
    'save_period': 10,          # Save checkpoint every N epochs
    'device': device,           # Device to use
    'project': 'runs/detect',   # Project directory
    'name': 'food_detection',   # Experiment name
    'exist_ok': True,           # Overwrite existing experiment
}

print("Training Configuration:")
print("="*50)
for key, value in TRAINING_CONFIG.items():
    print(f"{key}: {value}")
print("="*50)


tima

In [ ]:
# Start training
print("🚀 Starting training...")
print(f"Dataset: {DATASET_PATH}")
print(f"Config: {config_path}")
print(f"Model: {MODEL_SIZE}")
print("\nThis may take a while. Progress will be shown below...\n")

# Train the model
results = model.train(
    data=config_path,           # Path to data.yaml
    epochs=TRAINING_CONFIG['epochs'],
    imgsz=TRAINING_CONFIG['imgsz'],
    batch=TRAINING_CONFIG['batch'],
    workers=TRAINING_CONFIG['workers'],
    lr0=TRAINING_CONFIG['lr0'],
    lrf=TRAINING_CONFIG['lrf'],
    momentum=TRAINING_CONFIG['momentum'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    hsv_h=TRAINING_CONFIG['hsv_h'],
    hsv_s=TRAINING_CONFIG['hsv_s'],
    hsv_v=TRAINING_CONFIG['hsv_v'],
    degrees=TRAINING_CONFIG['degrees'],
    translate=TRAINING_CONFIG['translate'],
    scale=TRAINING_CONFIG['scale'],
    flipud=TRAINING_CONFIG['flipud'],
    fliplr=TRAINING_CONFIG['fliplr'],
    mosaic=TRAINING_CONFIG['mosaic'],
    mixup=TRAINING_CONFIG['mixup'],
    patience=TRAINING_CONFIG['patience'],
    save=TRAINING_CONFIG['save'],
    save_period=TRAINING_CONFIG['save_period'],
    device=TRAINING_CONFIG['device'],
    project=TRAINING_CONFIG['project'],
    name=TRAINING_CONFIG['name'],
    exist_ok=TRAINING_CONFIG['exist_ok'],
)

print("\n✅ Training completed!")
print(f"Results saved to: runs/detect/{TRAINING_CONFIG['name']}")


## Step 8: View Training Results

Analyze the training metrics and visualizations.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Path to results
results_path = Path(f"runs/detect/{TRAINING_CONFIG['name']}")

print("📊 Training Results")
print("="*50)

# Display training curves
results_dir = results_path
if (results_dir / "results.png").exists():
    print("\nTraining Curves:")
    display(Image(str(results_dir / "results.png")))
    
if (results_dir / "confusion_matrix.png").exists():
    print("\nConfusion Matrix:")
    display(Image(str(results_dir / "confusion_matrix.png")))

# Show best model path
best_model_path = results_dir / "weights" / "best.pt"
if best_model_path.exists():
    print(f"\n✅ Best model saved to: {best_model_path}")
    print(f"📦 Model size: {best_model_path.stat().st_size / 1e6:.2f} MB")
else:
    print(f"\n⚠️ Best model not found. Check: {results_dir}")

# Show last model path
last_model_path = results_dir / "weights" / "last.pt"
if last_model_path.exists():
    print(f"📦 Last checkpoint: {last_model_path}")


## Step 9: Test the Trained Model

Test your trained model on validation/test images.


In [ ]:
# Load the best trained model
best_model_path = f"runs/detect/{TRAINING_CONFIG['name']}/weights/best.pt"

if Path(best_model_path).exists():
    trained_model = YOLO(best_model_path)
    print(f"✅ Loaded trained model from: {best_model_path}")
    
    # Test on validation set
    print("\n🔍 Running validation...")
    validation_results = trained_model.val(
        data=config_path,
        imgsz=640,
        batch=16,
        conf=0.25,  # Confidence threshold
        iou=0.45,   # IoU threshold for NMS
    )
    
    print("\n📈 Validation Metrics:")
    print(f"mAP50: {validation_results.box.map50:.4f}")
    print(f"mAP50-95: {validation_results.box.map:.4f}")
    print(f"Precision: {validation_results.box.mp:.4f}")
    print(f"Recall: {validation_results.box.mr:.4f}")
else:
    print(f"⚠️ Best model not found at: {best_model_path}")
    print("Please check if training completed successfully.")


## Step 10: Test on Sample Images

Test your trained model on individual images.


In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from pathlib import Path

# Load trained model
best_model_path = f"runs/detect/{TRAINING_CONFIG['name']}/weights/best.pt"

if Path(best_model_path).exists():
    model = YOLO(best_model_path)
    
    # Test image path (update this with your test image)
    test_image_path = "/content/test_image.jpg"  # Update this path
    
    # If test image doesn't exist, try to get one from validation set
    val_images_dir = Path(DATASET_PATH) / "val" / "images"
    if not Path(test_image_path).exists() and val_images_dir.exists():
        test_images = list(val_images_dir.glob("*.jpg")) + list(val_images_dir.glob("*.png"))
        if test_images:
            test_image_path = str(test_images[0])
            print(f"Using validation image: {test_image_path}")
    
    if Path(test_image_path).exists():
        # Run inference
        results = model(test_image_path, conf=0.25, iou=0.45)
        
        # Display results
        for r in results:
            # Show image with detections
            im_array = r.plot()  # Plot a BGR numpy array of predictions
            cv2_imshow(im_array)
            
            # Print detection details
            print(f"\n🔍 Detections found: {len(r.boxes)}")
            for box in r.boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                cls_name = FOOD_CLASSES[cls_id] if cls_id < len(FOOD_CLASSES) else f"class_{cls_id}"
                print(f"  - {cls_name}: {conf:.2%} confidence")
    else:
        print(f"⚠️ Test image not found at: {test_image_path}")
        print("Please upload a test image or update test_image_path")
else:
    print(f"⚠️ Trained model not found. Please complete training first.")


## Step 11: Download Your Trained Model

Download the trained model to use in the Food Vision System.


In [ ]:
from google.colab import files
from pathlib import Path

# Path to best model
best_model_path = f"runs/detect/{TRAINING_CONFIG['name']}/weights/best.pt"

if Path(best_model_path).exists():
    print(f"📥 Downloading model: {best_model_path}")
    files.download(best_model_path)
    print("\n✅ Model downloaded! You can now use it in the Food Vision System.")
    print(f"\nTo use in FoodVisionSystem, update the model path:")
    print(f"  food_vision = FoodVisionSystem(model_name='path/to/best.pt')")
else:
    print(f"⚠️ Model not found at: {best_model_path}")
    print("Make sure training completed successfully.")


## Step 12: Integration with Food Vision System

Once you have your trained model, integrate it with the Food Vision System.

### Option A: Use in FoodVisionSystem.ipynb

1. Upload your trained `best.pt` model to Colab
2. Update the FoodVisionSystem initialization:
   ```python
   food_vision = FoodVisionSystem(model_name='path/to/your/best.pt')
   ```
3. Update the food_classes list to match your training classes:
   ```python
   self.food_classes = FOOD_CLASSES  # Your trained classes
   ```

### Option B: Export and Use Locally

1. Download the model to your computer
2. Use it with the Python script version
3. Ensure the nutrition database has entries for all your food classes
